# IMSE 441 — EDA Script (Engineering Perspective)
### Aligned to the 7-step EDA Roadmap
**Dataset expected:** `engineering_eda_dataset_v2.csv` (upload to Colab or place in Drive).

**Teaching note:** This notebook performs *EDA-first* (detect/diagnose) and keeps missing-data *treatment* in a separate Post‑EDA section.

## 0) Setup — Imports (all at once)
Run this cell once at the beginning.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from scipy.stats import entropy as shannon_entropy
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 120)

DATA_PATH = "engineering_eda_dataset_v2.csv"

KPV_COLS = [
    "setup_time_min", "cycle_time_min", "wait_time_min",
    "temperature_F", "pressure_bar", "vibration_mm_s",
    "torque_Nm", "humidity_pct", "energy_kwh",
]


## Step 1 — Understand the data structure
- What are the observations and variables?
- What are the data types and units?

**Interpretation:** Clarify what a row represents (e.g., one production run/job). Confirm `timestamp` and categorical identifiers are correct.

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])

print("Shape (rows, columns):", df.shape)
display(df.head(5))
display(df.dtypes)

if "timestamp" in df.columns:
    print("Time range:", df["timestamp"].min(), "to", df["timestamp"].max())

SENSOR_COLS = [c for c in df.columns if c.startswith("sensor_")]
print("Detected sensor columns:", len(SENSOR_COLS))


## Step 2 — Assess data quality
- Missing values (detect & diagnose)
- Duplicates
- Obvious inconsistencies / plausibility checks

**Potential actions:**
- If duplicates > 0, investigate ETL/merge logic.
- If missingness clusters by machine/shift/time, treat as systematic and justify any imputation.
- Flag physically impossible values for cleaning rules or domain review.

In [ ]:
# 2A) Duplicates
dup_count = int(df.duplicated().sum())
print("Duplicate rows:", dup_count)

# 2B) Missing values (detection)
missing_count = df.isna().sum()
missing_rate = df.isna().mean().sort_values(ascending=False)

display(missing_count[missing_count > 0].sort_values(ascending=False))
display((missing_rate[missing_rate > 0] * 100).round(2))

# 2C) Missingness patterns (diagnosis)
for grp in ["machine", "shift", "line", "product_type"]:
    if grp in df.columns and df[grp].nunique(dropna=True) > 1 and missing_rate.max() > 0:
        print(f"\nMissing rate by {grp} (top 8 columns with most missing overall):")
        top_missing_cols = missing_rate.head(8).index.tolist()
        display(df.groupby(grp)[top_missing_cols].apply(lambda x: x.isna().mean()).sort_index())

# 2D) Plausibility checks
if "vibration_mm_s" in df.columns:
    neg_vib = int((df["vibration_mm_s"] < 0).sum(skipna=True))
    print("Negative vibration values (physically suspicious):", neg_vib)

if "defect" in df.columns and "defect_count" in df.columns:
    inconsistent = int(((df["defect"] == 1) & (df["defect_count"] == 0)).sum())
    print("Rows where defect==1 but defect_count==0:", inconsistent)

existing_kpv = [c for c in KPV_COLS if c in df.columns]
if existing_kpv:
    display(df[existing_kpv].describe(include="all"))


## Step 3 — Summarize key variables
- Central tendency (mean, median)
- Variability (range, IQR, std)

**Interpretation:** For skewed processes, prefer median/IQR for “typical” behavior; use std to discuss stability.

In [ ]:
def robust_summary(series: pd.Series) -> pd.Series:
    s = series.dropna()
    return pd.Series({
        "n": len(s),
        "mean": s.mean(),
        "median": s.median(),
        "std": s.std(),
        "Q1": s.quantile(0.25),
        "Q3": s.quantile(0.75),
        "IQR": s.quantile(0.75) - s.quantile(0.25),
        "min": s.min(),
        "max": s.max(),
        "p90": s.quantile(0.90),
        "p95": s.quantile(0.95),
        "p99": s.quantile(0.99),
    })

existing_kpv = [c for c in KPV_COLS if c in df.columns]
if existing_kpv:
    display(df[existing_kpv].describe())

for c in ["cycle_time_min", "wait_time_min", "setup_time_min"]:
    if c in df.columns:
        print(f"\nRobust summary for {c}:")
        display(robust_summary(df[c]))

for grp in ["shift", "machine"]:
    if grp in df.columns and "cycle_time_min" in df.columns:
        print(f"\nGroup summary by {grp}:")
        cols = [c for c in ["cycle_time_min", "wait_time_min"] if c in df.columns]
        display(df.groupby(grp)[cols].agg(["count", "mean", "std", "median"]))


## Step 4 — Examine distributions and outliers
- Skewness and tails
- Error vs rare events

**Potential actions:**
- If outliers correspond to disruptions (failures/defects), analyze separately as special-cause.
- If outliers are measurement errors, define cleaning rules (clip, set to NaN, remove).

In [ ]:
# 4A) Distributions
for c in ["cycle_time_min", "wait_time_min"]:
    if c in df.columns:
        plt.figure()
        df[c].dropna().hist(bins=30)
        plt.title(f"Histogram: {c}")
        plt.xlabel(c)
        plt.ylabel("Frequency")
        plt.show()
        print(f"{c} skewness:", float(df[c].dropna().skew()))

# 4B) IQR outliers (both low + high)
def iqr_outliers(df_in: pd.DataFrame, col: str, k: float = 1.5):
    x = df_in[col].dropna()
    q1, q3 = x.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    out = df_in[(df_in[col] < lower) | (df_in[col] > upper)].copy()
    out["outlier_type"] = np.where(out[col] < lower, "low", "high")
    return out, float(lower), float(upper)

for c in ["cycle_time_min", "wait_time_min", "vibration_mm_s"]:
    if c in df.columns:
        out, lo, hi = iqr_outliers(df, c, k=1.5)
        print(f"\nIQR bounds for {c}: lower={lo:.3f}, upper={hi:.3f} | outliers={len(out)}")
        display(out[[x for x in ["timestamp","machine","shift","line","product_type",c,"outlier_type"] if x in out.columns]].head(10))

# 4C) Boxplots
for c in ["cycle_time_min", "wait_time_min"]:
    if c in df.columns:
        plt.figure()
        df.boxplot(column=c)
        plt.title(f"Boxplot: {c}")
        plt.ylabel(c)
        plt.show()


## Step 5 — Explore relationships and redundancy
- Correlation
- Scatter plots
- Multicollinearity intuition

**Interpretation:** High correlation suggests redundancy; scatter patterns can reveal nonlinear effects or regime changes.

In [ ]:
num_cols = df.select_dtypes(include="number").columns.tolist()
display(df[num_cols].corr())

focus = [c for c in ["cycle_time_min","wait_time_min","temperature_F","pressure_bar","vibration_mm_s","torque_Nm","energy_kwh"] if c in df.columns]
if len(focus) >= 2:
    display(df[focus].corr())

scatter_pairs = [
    ("cycle_time_min", "vibration_mm_s"),
    ("cycle_time_min", "temperature_F"),
    ("cycle_time_min", "pressure_bar"),
]
for x, y in scatter_pairs:
    if x in df.columns and y in df.columns:
        plt.figure()
        plt.scatter(df[x], df[y], s=10, alpha=0.35)
        plt.title(f"Scatter: {x} vs {y}")
        plt.xlabel(x)
        plt.ylabel(y)
        plt.show()

if SENSOR_COLS:
    sensor_corr = df[SENSOR_COLS].corr()
    upper = sensor_corr.where(np.triu(np.ones(sensor_corr.shape), k=1).astype(bool))
    top_pairs = upper.abs().stack().sort_values(ascending=False).head(15)
    print("\nTop correlated sensor pairs (possible redundancy):")
    display(top_pairs)


## Step 6 — Check class balance and categorical structure
- Rare events (defects, failures)
- Category coverage and entropy

**Potential actions:**
- If rare-event: plan to use recall/precision and cost-aware evaluation later.
- If some categories are tiny: avoid over-interpreting subgroup rates.
- Entropy helps summarize category dominance vs diversity.

In [ ]:
for y in ["failure", "defect"]:
    if y in df.columns:
        print(f"\nClass balance for {y}:")
        counts = df[y].value_counts(dropna=False).sort_index()
        props = (counts / len(df) * 100).round(2)
        display(pd.DataFrame({"count": counts, "percent": props}))

for c in ["line", "machine", "shift", "product_type"]:
    if c in df.columns:
        print(f"\nCategory counts: {c}")
        display(df[c].value_counts())

for grp in ["shift", "machine", "line", "product_type"]:
    if grp in df.columns and "defect" in df.columns:
        print(f"\nDefect rate by {grp}:")
        display(pd.crosstab(df[grp], df["defect"], normalize="index"))

def entropy_bits(series: pd.Series) -> float:
    p = series.value_counts(normalize=True, dropna=True)
    return float(shannon_entropy(p, base=2))

for c in ["shift", "machine", "product_type"]:
    if c in df.columns:
        k = df[c].nunique(dropna=True)
        H = entropy_bits(df[c])
        Hmax = np.log2(k) if k > 1 else 0.0
        print(f"Entropy H({c}) = {H:.3f} bits (max ≈ {Hmax:.3f} for {k} categories)")

if "failure" in df.columns:
    y_true = df["failure"].fillna(0).astype(int).values
    y_pred = np.zeros_like(y_true)
    print("\nAccuracy paradox demo (always predict no failure):")
    print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred, zero_division=0))
    print("Recall:", recall_score(y_true, y_pred, zero_division=0))


## Step 7 — Evaluate dimensionality and scaling needs
- Need for normalization
- Dimensionality reduction (PCA)

**Interpretation:** PCA is scale-sensitive; standardize first. If a few PCs explain most variance, sensors are redundant and the system may be low-dimensional.

In [ ]:
pca_features = []
pca_features += SENSOR_COLS
pca_features += [c for c in ["temperature_F","pressure_bar","vibration_mm_s","torque_Nm","humidity_pct"] if c in df.columns]
pca_features = list(dict.fromkeys([c for c in pca_features if c in df.columns]))

if len(pca_features) >= 3:
    X = df[pca_features].copy()

    # Simple median imputation for PCA to keep rows (justify based on Step 2 missingness diagnosis)
    X = X.apply(lambda s: s.fillna(s.median()), axis=0)

    Xs = StandardScaler().fit_transform(X)

    pca = PCA()
    pca.fit(Xs)

    evr = pca.explained_variance_ratio_
    cum = np.cumsum(evr)

    print("PCA cumulative explained variance (first 10):")
    display(pd.Series(cum[:10], index=[f"PC{i}" for i in range(1, 11)]))

    n90 = int(np.argmax(cum >= 0.90) + 1)
    n95 = int(np.argmax(cum >= 0.95) + 1)
    print(f"Components for 90% variance: {n90}")
    print(f"Components for 95% variance: {n95}")

    plt.figure()
    plt.plot(cum, marker="o")
    plt.title("PCA: Cumulative Explained Variance")
    plt.xlabel("Number of components")
    plt.ylabel("Cumulative explained variance")
    plt.show()

    Z = pca.transform(Xs)[:, :2]
    plt.figure()
    plt.scatter(Z[:, 0], Z[:, 1], s=10, alpha=0.35)
    plt.title("PCA Projection: PC1 vs PC2")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.show()
else:
    print("Not enough PCA features found (need at least 3). Check sensor column names.")


## Post‑EDA (Optional) — Missing value treatment examples
This section is intentionally separated from EDA.

**Teaching note:** In EDA we detect/diagnose missingness early; treatment decisions require assumptions (MCAR/MAR/MNAR).

In [ ]:
# EXAMPLE A: Drop rows (only if missingness is small and plausibly random)
# df_drop = df.dropna(subset=["temperature_F"])

# EXAMPLE B: Global median imputation (simple, robust, may reduce variance artificially)
# df_imp = df.copy()
# df_imp["temperature_F"] = df_imp["temperature_F"].fillna(df_imp["temperature_F"].median())

# EXAMPLE C: Groupwise imputation (often appropriate in manufacturing: machine-specific baselines)
# df_imp["pressure_bar"] = df_imp.groupby("machine")["pressure_bar"].transform(lambda s: s.fillna(s.median()))
